Dogs vs Cats

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/AI 1/assignment3/modeller', exist_ok=True)
print("Drive kopplad och mappar skapade!")

Mounted at /content/drive
Drive kopplad och mappar skapade!


In [ ]:
import os
import json

os.makedirs('/root/.kaggle', exist_ok=True)


with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API klar!")

Kaggle API klar!


In [ ]:
import kagglehub
import os

# Ladda ner datasetet
path = kagglehub.dataset_download("blourdhuraju/dogs-vs-cats-classification")
print("Path to dataset files:", path)

# Kolla vad som finns i mappen
for item in os.listdir(path):
    print(item)

Using Colab cache for faster access to the 'dogs-vs-cats-classification' dataset.
Path to dataset files: /kaggle/input/dogs-vs-cats-classification
dogs-vs-cats-classification


In [ ]:
import os

inner_path = '/kaggle/input/dogs-vs-cats-classification/dogs-vs-cats-classification'

# Kolla antal bilder
for folder in ['train', 'test', 'validation']:
    for animal in ['dogs', 'cats']:
        path = os.path.join(inner_path, folder, ani mal)
        antal = len(os.listdir(path))
        print(f"{folder}/{animal}: {antal} bilder")

train/dogs: 9966 bilder
train/cats: 9977 bilder
test/dogs: 1247 bilder
test/cats: 1248 bilder
validation/dogs: 1245 bilder
validation/cats: 1247 bilder


1. Importera bibliotek och förbereda datan. Ladda in hundarna och katterna för att förbereda dem för tärning med data augmentation.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os

# Sökväg till datan
data_path = '/kaggle/input/dogs-vs-cats-classification/dogs-vs-cats-classification'

# Förbereda bilderna för träning (med augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),      # ändra storlek till 224x224
    transforms.RandomHorizontalFlip(),  # spegelvänd slumpmässigt
    transforms.RandomRotation(10),      # rotera lite
    transforms.ToTensor(),              # omvandla till tal
    transforms.Normalize(               # normalisera
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

# Förbereda testbilder (ingen augmentation!)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

# Ladda in datan
train_data = torchvision.datasets.ImageFolder(
    os.path.join(data_path, 'train'),
    transform=train_transform)

test_data = torchvision.datasets.ImageFolder(
    os.path.join(data_path, 'test'),
    transform=test_transform)

val_data = torchvision.datasets.ImageFolder(
    os.path.join(data_path, 'validation'),
    transform=test_transform)

# Skapa data loaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)
val_loader = DataLoader(val_data, batch_size=32)

print(f"Träningsbilder:     {len(train_data)}")
print(f"Testbilder:         {len(test_data)}")
print(f"Valideringsbilder:  {len(val_data)}")

Träningsbilder:     19943
Testbilder:         2495
Valideringsbilder:  2492


2. Bygg ett eget CNN för hund vs katt.
Vi bygger ett från grunden som vi gjorde med MNIST.

In [ ]:
import torch
import torch.nn as nn

class CNN_HundKatt(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Första lagret
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            # Andra lagret
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            # Tredje lagret
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)  # 2 klasser: hund eller katt
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Använder: {device}")

model = CNN_HundKatt().to(device)
print("CNN modell skapad!")

Använder: cuda
CNN modell skapad!


3. Träna CNN på hund vs katt
Tränar nätverket och mäter noggranheten efter varje epoch.   

In [ ]:
import time
import os

os.makedirs('/content/AI 1/assignment3/modeller', exist_ok=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(10):
    epoch_start = time.time()

    # Träning
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    # Mät noggrannhet på valideringsdata
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    # Spara modellen
    torch.save(model.state_dict(),
        f'/content/drive/MyDrive/AI 1/assignment3/modeller/hundkatt_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")

Epoch 1 → Noggrannhet: 65.61% | Tid: 186.1s
Epoch 2 → Noggrannhet: 64.25% | Tid: 118.6s
Epoch 3 → Noggrannhet: 62.92% | Tid: 117.3s
Epoch 4 → Noggrannhet: 62.44% | Tid: 116.0s
Epoch 5 → Noggrannhet: 70.51% | Tid: 113.2s
Epoch 6 → Noggrannhet: 64.53% | Tid: 114.9s
Epoch 7 → Noggrannhet: 70.10% | Tid: 116.5s
Epoch 8 → Noggrannhet: 76.12% | Tid: 118.4s
Epoch 9 → Noggrannhet: 77.37% | Tid: 118.7s
Epoch 10 → Noggrannhet: 78.77% | Tid: 117.7s

Träning klar! Total tid: 1244.6s


4. Transfer learning med ResNet50
Vi använder ResNet50 som är redan tränas på miljoner bilder, vi anpassar bara sista lagret till vår träningsdata.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

# Ladda in det färdiga ResNet50 nätverket
model_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

# Frys alla lager - vi vill inte ändra dem!
for param in model_resnet.parameters():
    param.requires_grad = False

# Byt ut sista lagret till vårt eget (hund vs katt)
model_resnet.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 2)  # 2 klasser: hund eller katt
)

model_resnet = model_resnet.to(device)
print("ResNet50 modell klar!")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


ResNet50 modell klar!


Träna på ResNet på hund vs katt, vi tränar bara på sista lagret.

In [ ]:
import time
import os

optimizer_resnet = torch.optim.Adam(
    model_resnet.fc.parameters(),  # träna bara sista lagret!
    lr=0.001
)
criterion = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(10):
    epoch_start = time.time()

    # Träning
    model_resnet.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer_resnet.zero_grad()
        output = model_resnet(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_resnet.step()

    # Mät noggrannhet
    model_resnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model_resnet(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    noggrannhet = 100 * correct / total
    epoch_tid = time.time() - epoch_start
    print(f"Epoch {epoch+1} → Noggrannhet: {noggrannhet:.2f}% | Tid: {epoch_tid:.1f}s")

    # Spara modellen
    torch.save(model_resnet.state_dict(),
        f'/content/drive/MyDrive/AI 1/assignment3/modeller/resnet_epoch_{epoch+1}.pth')

total_tid = time.time() - total_start
print(f"\nTräning klar! Total tid: {total_tid:.1f}s")


Epoch 1 → Noggrannhet: 98.52% | Tid: 132.0s
Epoch 2 → Noggrannhet: 98.27% | Tid: 126.0s
Epoch 3 → Noggrannhet: 98.56% | Tid: 125.6s
Epoch 4 → Noggrannhet: 98.68% | Tid: 124.5s
Epoch 5 → Noggrannhet: 98.23% | Tid: 123.4s
Epoch 6 → Noggrannhet: 98.76% | Tid: 123.6s
Epoch 7 → Noggrannhet: 98.76% | Tid: 123.9s
Epoch 8 → Noggrannhet: 98.76% | Tid: 125.8s
Epoch 9 → Noggrannhet: 98.68% | Tid: 124.9s
Epoch 10 → Noggrannhet: 98.68% | Tid: 122.9s

Träning klar! Total tid: 1255.4s
